# SST Checkpointing Comparison Workflow

# Configuration

## Import workflows module

In [ ]:
from utils.workflows import *

## Global params

Users can modify these top-level parameters to alter the behavior of this workflow.

In [ ]:
# ---------------------------------------------------------------------------------------------------------------------
# !!! DO NOT MODIFY THE CODE BELOW   !!!
# !!!  (Modify in the next section)  !!!
# ---------------------------------------------------------------------------------------------------------------------

# So that can you can maintain the defaults, we suggest you don't directly edit
# the parameters inline here but rather overwite values at the bottom of this
# cell.

# We'll store our containers and benchmark results under the specified directory
# (it will be created if it doesn't already exist).
import os
if 'user_customExperimentsDir' in globals():
    baseDir = f'{user_customExperimentsDir}/checkpointing'
else:
    baseDir=f'{os.getenv("HOME")}/workflows/checkpointing'

# Run using an SST in the specified container. To find containers to use see the
# container factory at https://github.com/hpc-ai-adv-dev/sst-container-factory
# Prebuild containers are available at https://github.com/orgs/hpc-ai-adv-dev/packages
container_url  = 'ghcr.io/hpc-ai-adv-dev/sst-core:master-latest'
container_name = None # DO NOT MODIFY THIS LINE: Variable will be assigned after we download the container
                      # We include it here to document what global variables are available throughout the
                      # notebook.

# The benchmark will be cloned from the specified repository. We assume the
# benchmark itself is in the 'benchmarkPath' directory within the repos.  We
# assume building the benchmark is a matter of running 'make' in that directoy.
benchmarkRepos='https://github.com/hpc-ai-adv-dev/sst-benchmarks.git'
benchmarkPath='phold'

# Run the benchmark with varying numbers of components
m = 1_000_000
num_comps_per_trial = [m, m*2, m*3, m*4, m*5, m*6, m*7, m*8]

jobs = [
 # suite        depends on    args
 ('baseline',   None,         '--print-timing-info=3 --parallel-load=SINGLE ./phold_dist.py -- --width {size} --height {size} --event-density 0.0 --time-to-run 1ns'),
 ('checkpoint', None,         '--print-timing-info=3 --timing-info-json=time_checkpoint_{size} --checkpoint-sim-period=1ns --checkpoint-prefix=checkpt_{size} --parallel-load=SINGLE ./phold_dist.py -- --width {size} --height {size} --event-density 0.0 --time-to-run 1ns'),
 ('restore',    'checkpoint', '--print-timing-info=3 --timing-info-json=time_restore_{size} --load-checkpoint ./checkpt_{size}/checkpt_{size}_1_1000/checkpt_{size}_1_1000.sstcpt')]

# Additional arguments to pass when launching jobs with srun. For example the
# partition name or --qos=high for higher priority in the queue.
additional_srun_args = ''

# Several of the setup steps will avoid rerunning if they have previously been run. Append to this
# list to indicate when you want to force a step to be reproduced.
#
# VALID VALUES ARE:
#   'ALL'     
#   'DOWNLOAD_CONTAINERS' 
#   'DOWNLOAD_BENCHMARKS' 
#   'BUILD_BENCHMARKS'      Note: we always rerun make, if this is set we will also run 'make clean' before rebuilding
force = []

# ---------------------------------------------------------------------------------------------------------------------
# Overwite parameters below this line to customize the workflow: 
# ---------------------------------------------------------------------------------------------------------------------

m = 100_000
num_comps_per_trial = [m, m*2, m*3, m*4, m*5, m*6, m*7, m*8]

## Environment

In [ ]:
set_workflow_log(f'{baseDir}/workflow.log')
run_cmd(f'e4s-cl profile edit --add-files {baseDir}')

## Download containers

In [ ]:
_force = 'ALL' in force or 'DOWNLOAD_CONTAINERS' in force

container_name = download_custom_container(container_url, force=_force)

## Download benchmarks

In [ ]:
_force = 'ALL' in force or 'DOWNLOAD_BENCHMARKS' in force

if not os.path.exists(f'benchmarks') or _force:
    run_cmd(f"git clone {benchmarkRepos} benchmarks")
    run_cmd(f"e4s-cl profile edit --add-files {baseDir}/benchmarks/{benchmarkPath}")
else:
    print(f"Benchmarks from {benchmarkRepos} have already been downloaded, skipping download.")

## Build benchmarks 

In [ ]:
_force = 'ALL' in force or 'BUILD_BENCHMARKS' in force

cd(f"{baseDir}/benchmarks/phold")
run_cmd('touch sstsimulator.conf')
_cmd = 'make' if not _force else 'make clean; make'
run_in_container(_cmd,
    f'{baseDir}/{container_name}',
    additional_apptainer_args=f'--bind sstsimulator.conf:{os.getenv("HOME")}/.sst/sstsimulator.conf')
cd(baseDir)

# Run

## Start jobs

In [ ]:
import math
import shutil
import subprocess
import threading
import time
from io import StringIO
import os

runDisplay = SafeDisplay(display_handle = display('', display_id="run_disp"))

# Prepare run directories (remove any old copies, make a fresh new one)
cd(f'{baseDir}/benchmarks/phold')

# List to track all log threads
threads = {}
for run in jobs:
    suite      = run[0]
    depends_on = run[1]
    sst_args_template   = run[2]
    run_dir = f'{baseDir}/runs/{suite}'
    threads[suite] = {}

    # Remove old results
    if os.path.exists(run_dir):
        shutil.rmtree(run_dir)
    os.makedirs(run_dir, exist_ok=True)

    for size_sqrd in num_comps_per_trial:
        size = int(math.sqrt(size_sqrd))
        sst_args = sst_args_template.format(size=size)

        depends_on_thread = None
        if depends_on is not None:
            depends_on_thread = threads[depends_on][size]

        threads[suite][size] = \
            launch_and_log_sst(
                image        = f'{baseDir}/{container_name}',
                srun_args    = f'-N 1 --job-name={suite}_{size}',
                sst_args     = sst_args,
                log_file     = f'{run_dir}/size_{size}',
                config_path  = f'{baseDir}/benchmarks/phold/sstsimulator.conf',
                safe_display = runDisplay,
                depends_on   = depends_on_thread)

## Watch squeue

In [ ]:
watch_queue_widget()

## Inspect results

In [ ]:
inspect_logs(f'{baseDir}/runs')

# Preprocess

In [ ]:
import os
from pathlib import Path

for job in jobs:
    run_path = f"{baseDir}/runs/{job[0]}"
    data = extract_sst_output_in_files(Path(run_path).iterdir())
    csv_lines = convert_to_csv(data)
    csv_name = f"{baseDir}/runs/{job[0]}.csv"
    with open(csv_name, 'w') as f:
        f.write('\n'.join(csv_lines))

    if os.path.exists(csv_name):
        with open(csv_name, 'r') as f:
            print(f'\n===== {csv_name} =====')
            print(f.read())
    else:
        print(f'\n===== {csv_name} (not created) =====')

# Plot

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

class Style:
    RED = '\033[91m'
    RESET = '\033[0m'

try:
    df_baseline   = pd.read_csv(f"{baseDir}/runs/baseline.csv")
    df_checkpoint = pd.read_csv(f"{baseDir}/runs/checkpoint.csv")
    df_restore    = pd.read_csv(f"{baseDir}/runs/restore.csv")
except FileNotFoundError as e:
    display(HTML(f"<span style='background-color:red'>ERROR</span>: File not found - {e.filename}"))
    raise StopExecution()

for var, suffix in [("df_baseline", "baseline"),
                    ("df_checkpoint", "checkpoint"),
                    ("df_restore", "restore")]:
    globals()[var] = globals()[var].rename(
        columns=lambda c: f"{c}_{suffix}" if c != "Size" else c
    )


df = pd.merge(df_baseline, df_checkpoint, on="Size")
df = pd.merge(df, df_restore, on="Size")
#print(df)

fig = plt.figure()
ax = fig.add_subplot(111)

ax.scatter(x=df["Size"], y=df['total_duration_baseline'],   c='b', marker="s", label='Baseline')
ax.scatter(x=df["Size"], y=df['total_duration_checkpoint'], c='g', marker="o", label='Checkpoint')
ax.scatter(x=df["Size"], y=df['total_duration_restore'],    c='r', marker="o", label='Restore')
ax.ticklabel_format(style='sci', scilimits=(6, 6), axis='x')
plt.title('SST PHOLD 1ns run checkpoint vs restore time')
plt.xlabel('Millions of PHOLD components')
plt.ylabel('Time (s)')
plt.legend(loc='upper left')
plt.show()